# Week 10 — Day 2 — Session 3

## Making Convolution Intuitive

**Main Goal:** Consolidate and deepen the concepts from Day 1 rather than introducing many new concepts.

> **Core idea of this session:**
> A CNN is not a mysterious new type of neural network. It is a neural network that learns **local visual patterns** using learnable filters and combines those patterns into increasingly useful representations.

---

# Learning Objectives

By the end of this session, students should be able to:

1. Explain the complete CNN pipeline at a conceptual level.
2. Perform a small convolution operation by hand.
3. Explain the difference between **kernel, filter, and feature map**.
4. Explain how classical filters such as Sobel are related to CNN filters.
5. Explain the effect of **stride** and **padding**.
6. Explain the concept of **receptive field**.
7. Explain why deeper CNN layers can represent increasingly complex patterns.
8. Calculate basic feature-map dimensions.

---

# 1. Reconnect to the Big Picture


> **What happens when we give an image to a CNN?**


A simplified pipeline is:

```text
Image
  ↓
Convolution
  ↓
Feature Maps
  ↓
Activation
  ↓
Pooling
  ↓
More Convolution
  ↓
More Features
  ↓
Classification
```

---

## 1.1 The Important Question


> **What is the CNN actually learning?**

The answer is:

$$
\boxed{\text{The weights}}
$$

But in a CNN, some of those weights are organized as **convolutional filters**.

Therefore:

```text
 Neural Network
      ↓
Learnable Weights


          CNN
           ↓
Learnable Convolutional Filters
```

---

# 2. A CNN Starts With an Image Tensor

For a grayscale image:

$$
H\times W
$$

For example:

$$
28\times28
$$

For an RGB image:

$$
H\times W\times3
$$

For example:

$$
32\times32\times3
$$

In PyTorch, an individual RGB image is usually represented as:

```text
[C, H, W]
```

So CIFAR-10 becomes:

```text
[3, 32, 32]
```

With a batch of 64 images:

```text
[64, 3, 32, 32]
```

where:

* `64` → batch size
* `3` → RGB channels
* `32` → height
* `32` → width

---

# 3. Convolution by Hand


Consider:

$$
I=
\begin{bmatrix}
1&2&3&4&5\\
1&2&3&4&5\\
1&2&3&4&5\\
1&2&3&4&5\\
1&2&3&4&5
\end{bmatrix}
$$

And a kernel:

$$
K=
\begin{bmatrix}
-1&0&1\\
-1&0&1\\
-1&0&1\
\end{bmatrix}
$$


> What happens if we put the kernel over the upper-left corner?

---

## 3.1 First Position

The first $3\times3$ region is:

$$
\begin{bmatrix}
1&2&3\\
1&2&3\\
1&2&3\
\end{bmatrix}
$$

Multiply element-by-element:

$$
\begin{aligned}
&1(-1)+2(0)+3(1)\\
+&1(-1)+2(0)+3(1)\\
+&1(-1)+2(0)+3(1)
\end{aligned}
$$

Therefore:

$$
-1+3-1+3-1+3=
6
$$

So the first output value is:

$$
\boxed{6}
$$

---

# 3.2 Move the Kernel

Now move the kernel one position to the right.

```text
Image

┌───────────┐
│ 1 2 3 4 5 │
│ 1 2 3 4 5 │
│ 1 2 3 4 5 │
│ 1 2 3 4 5 │
│ 1 2 3 4 5 │
└───────────┘
```

The new region is:

$$
\begin{bmatrix}
2&3&4\\
2&3&4\\
2&3&4
\end{bmatrix}
$$

The calculation becomes:

$$
2(-1)+3(0)+4(1)
$$

for each row.

Therefore:

$$
-2+4-2+4-2+4=6
$$

Again:

$$
\boxed{6}
$$

---

# 3.3 Repeat the Process

Continue sliding the kernel across the image.

Conceptually:

```text
   Image
     │
     ▼
┌─────────┐
│ Kernel  │
└─────────┘
     ↓
   Move
     ↓
┌─────────┐
│ Kernel  │
└─────────┘
     ↓
   Move
     ↓
┌─────────┐
│ Kernel  │
└─────────┘
```

Each position produces **one number**.

All those numbers form a new matrix:

```text
Input Image
     ↓
Convolution
     ↓
Feature Map
```

---

# 3.4 Why Convolution Uses Far Fewer Weights

Now let's compare a convolutional layer with a fully connected layer for the **same image**.

Suppose our input image is:

$$
32\times32\times3
$$

Therefore, the image contains:

$$
32\times32\times3
=3072
$$

input values.

---

## Fully Connected Layer
**assumption: bias = 0**
Suppose we connect every pixel/channel value to **one neuron**.

The neuron needs one weight for every input value:

$$
3072
$$

weights.

If we have 100 neurons:

$$
3072\times100
=307,200
$$

weights.

---

## Convolutional Layer

Now consider a convolutional layer with:

* Kernel size: $3\times3$
* Input channels: $3$
* Number of filters: $100$

Each filter contains:

$$
3\times3\times3
=27
$$

weights.

Therefore, the entire convolutional layer contains:

$$
27\times100
=2700
$$

weights.

Ignoring bias for simplicity:

| Layer           | Input Image         | Configuration          | Number of Weights |
| --------------- | ------------------- | ---------------------- | ----------------: |
| Fully Connected | $32\times32\times3$ | 1 neurons            |       **3072** |
| Convolutional   | $32\times32\times3$ | 1 filter, $3\times3$ |           **27** |
| Fully Connected | $32\times32\times3$ | 100 neurons            |       **307,200** |
| Convolutional   | $32\times32\times3$ | 100 filters, $3\times3$ |           **2700** |

So:

$$
\frac{307,200}{2700}
\approx125
$$

The fully connected layer has **about 125 times more weights** in this example.

---

## Why Is the Difference So Large?

The key idea is **weight sharing**.

A fully connected layer learns different weights for different locations:

```text
Pixel 1 ───────► Weight 1
Pixel 2 ───────► Weight 2
Pixel 3 ───────► Weight 3
       ...
Pixel 3072 ────► Weight 3072
```

A convolutional filter, however, uses the **same weights at every spatial location**:

```text
       Same Filter
           │
           ▼
Image → [3×3] → value
           │
           ▼
        move
           │
           ▼
Image → [3×3] → value
           │
           ▼
        move
           │
           ▼
Image → [3×3] → value
```

The same 27 weights are reused across the entire image.

This is called:

$$
\boxed{\text{Weight Sharing}}
$$

---

## The Deeper Intuition

A fully connected network effectively asks:

> "What should I learn about every individual pixel position?"

A convolutional layer asks:

> "What local visual pattern should I look for, regardless of where it appears?"

For example, if a filter learns to detect a vertical edge, we want to detect that edge whether it appears:

```text
Left side       Center        Right side

   │              │              │
   │              │              │
   │              │              │
```

The same filter can be used everywhere.

Therefore:

$$
\boxed{
\text{Local Connectivity}
+
\text{Weight Sharing}
=\text{Far Fewer Parameters}
}
$$

---

### Important Point

Do **not** say that CNNs have fewer weights simply because the kernel is small.

The deeper reason is:

1. **Local connectivity** — each output looks only at a small region.
2. **Weight sharing** — the same filter is reused across all spatial locations.

These two ideas are fundamental to why CNNs are effective for images.

> **A CNN learns a small set of useful visual detectors and reuses them across the entire image.**

# 3.4 The Key Idea

Emphasize:

> **A convolution does not produce just one number.**

The kernel is applied at many locations.

Each location produces one output value.

Therefore:

$$
\boxed{
\text{Many local calculations}
\rightarrow
\text{Feature Map}
}
$$

---

# 4. What Does the Kernel Detect?

Now return to the kernel:

$$
K=
\begin{bmatrix}
-1&0&1\\
-1&0&1\\
-1&0&1
\end{bmatrix}
$$

Ask:

> Why did we choose these numbers?

The kernel compares the left side with the right side.

Conceptually:

```text
Dark → Bright
```

A strong response means:

> There is a strong intensity transition in this region.

This is the basic idea behind an **edge detector**.

---

# 5. Kernel vs Filter vs Feature Map


This is an important source of confusion, so slow down here.

---

## 5.1 Kernel

A **kernel** is a small set of weights used in a convolution.

For a simple grayscale example:

$$
3\times3
$$

For example:

$$
\begin{bmatrix}
-1&0&1\\
-1&0&1\\
-1&0&1
\end{bmatrix}
$$

---

## 5.2 Filter

In CNN terminology, a filter usually refers to the complete set of kernel weights that operates across **all input channels**.

For an RGB image:

```text
Input:

3 × H × W
```

A filter with a $3\times3$ spatial kernel has:

```text
3 × 3 × 3
```

weights.

Conceptually:

```text
          RGB Image (3xHxW)
              │
      ┌───────┼───────┐
      ▼       ▼       ▼
    Kernel   Kernel   Kernel
      R        G        B
      └───────┼───────┘
              ▼
        One Feature Map (1xHxW)
```

---

## 5.3 One Filter Produces One Feature Map

This is the most important relationship:

$$
\boxed{
\text{One Filter}
\rightarrow
\text{One Feature Map}
}
$$

For example:

```text
Input:
3 × 32 × 32

Conv2D:
16 filters

Output:
16 × 32 × 32
```

Why?

Because:

$$
16\text{ filters}
\rightarrow
16\text{ feature maps}
$$

---

# 5.4 What If We Have 32 Filters?

Suppose:

```python
nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3
)
```Then the layer contains:

$$
32
$$

filters.

Each filter looks across all three RGB channels.

Therefore:

```text
RGB Image
   ↓
32 learned filters
   ↓
32 feature maps
```

---

# 5.5 A Useful Mental Model

Tell students:

> Think of each filter as a **visual question**.

For example:

```text
Filter 1:
"Is there a vertical edge here?"

Filter 2:
"Is there a horizontal edge here?"

Filter 3:
"Is there a particular color transition here?"

Filter 4:
"Is there a particular texture here?"
```

The network learns these questions automatically.

---

# 6. Classical Filters → CNN Filters


Now connect this to the classical computer vision material from Day 1.

Before deep learning, humans designed filters.

For example, the Sobel operator:

$$
G_x=
\begin{bmatrix}
-1&0&1\\
-2&0&2\\
-1&0&1
\end{bmatrix}
$$

This filter is designed to respond strongly to certain edges.

---

## 6.1 Classical Computer Vision

```text
Human
  ↓
Design Kernel
  ↓
Apply Kernel
  ↓
Feature / Edge Detection
```

For example:

```text
Sobel Kernel
     ↓
Edge Map
```

---

## 6.2 CNN

A CNN does something different:

```text
Training Data
      ↓
CNN
      ↓
Learn Filter Weights
      ↓
Feature Maps
```

The filter starts with initialized weights.

Training gradually modifies them:

$$
W_{\text{new}}
= W_{\text{old}}
-\eta
\frac{\partial L}{\partial W}
$$

Eventually, useful filters emerge.

---

# 6.3 Important Distinction

Do **not** say:

> "CNN filters are Sobel filters."

Instead:

> **Sobel is an excellent analogy for understanding convolutional filters, but CNN filters are learned from data.**

A CNN is not limited to edge detection.

It can learn:

* edges
* color transitions
* textures
* patterns
* shapes
* task-specific features

---

# 7. Stride

Now revisit stride visually.

https://sesen.ai/convolution-visualizer

### Stride = 1

The kernel moves one pixel at a time:

## Stride = 2

The kernel jumps two pixels.


Therefore:

> **Stride controls how far the kernel moves between two consecutive positions.**

---

# Effect on Output Size

The general formula is:

$$
O=
\left\lfloor
\frac{N+2P-K}{S}
\right\rfloor+1
$$

where:

* $N$ = input size
* $K$ = kernel size
* $P$ = padding
* $S$ = stride
* $O$ = output size

---

## Example

Suppose:

$$
N=7
$$

$$
K=3
$$

$$
P=0
$$

### Stride = 1

$$
O=
\left\lfloor
\frac{7-3}{1}
\right\rfloor+1
=5
$$

So:

$$
7\times7
\rightarrow
5\times5
$$

---

### Stride = 2

$$
O=
\left\lfloor
\frac{7-3}{2}
\right\rfloor+1
=3
$$

So:

$$
7\times7
\rightarrow
3\times3
$$

---

# 7.3 Key Intuition

Larger stride means:

```text
Larger stride
     ↓
Fewer kernel positions
     ↓
Smaller feature map
     ↓
More spatial downsampling
```


> **Stride controls how densely we scan the image.**

---

# 8. Padding


> What happens if we don't want the feature map to shrink?

We can add padding around the image.

For example:

```text
Original:

┌─────────┐
│         │
│ Image   │
│         │
└─────────┘
```

With padding:

```text
┌─────────────┐
│ 0 0 0 0 0   │
│ 0 Image 0   │
│ 0 Image 0   │
│ 0 0 0 0 0   │
└─────────────┘
```

The simplest padding uses zeros.

---

## 8.1 Why Use Padding?

For:

$$
K=3,\quad S=1
$$

if we use:

$$
P=1
$$

then:

$$
O=
\frac{N+2(1)-3}{1}+1
$$

which simplifies to:

$$
O=N
$$

Therefore:

```text
32 × 32
     ↓
Conv 3×3
Padding = 1
Stride = 1
     ↓
32 × 32
```

This is extremely common in CNNs.

---

# 9. Receptive Field


Now introduce the concept again, but connect it directly to convolution.

A neuron does not see the entire image.

It only sees a local region.

For a $3\times3$ convolution:

```text
Input

┌─────────────────┐
│                 │
│   ┌─────┐       │
│   │ 3×3 │       │
│   └─────┘       │
│                 │
└─────────────────┘
```

That region is the neuron's:

$$
\boxed{\text{Receptive Field}}
$$

---

# 9.1 Stacking Convolutions

Consider:

```text
Conv 3×3
    ↓
Conv 3×3
```

The first layer sees:

$$
3\times3
$$

But the second layer sees a $3\times3$ region of the **first feature map**.

Each of those neurons already sees a $3\times3$ region of the original image.

Therefore, the effective receptive field becomes:

$$
5\times5
$$

Conceptually:

```text
Layer 1

3 × 3
  ↓
Layer 2

5 × 5
```

---

<img src="res/rf.webp" width="400">

# 9.2 Add Another Layer

With three $3\times3$ convolutions:

$$
3\times3
\rightarrow
5\times5
\rightarrow
7\times7
$$

So:

> **As we go deeper, neurons can integrate information from larger regions of the original image.**

---

# 9.3 Receptive Field and Feature Hierarchy

This connects two important ideas:

```text
Small receptive field
        ↓
Local patterns
        ↓
Edges / simple textures
        ↓
Larger receptive field
        ↓
Shapes
        ↓
Object parts
        ↓
Larger context
        ↓
Complex visual patterns
```

This is one of the main reasons deep CNNs are powerful.

---

# 10. Mini Challenge


Given the following CNN:

```text
Input: 32 × 32 × 3

        ↓

Conv2D(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    stride=1,
    padding=1
)

        ↓ (16,32,32)

MaxPool2D(
    kernel_size=2,
    stride=2
)

        ↓ (16,16,16)

Conv2D(
    in_channels=16,
    out_channels=32,
    kernel_size=3,
    stride=1,
    padding=1
)

        ↓ (32,16,16)

MaxPool2D(
    kernel_size=2,
    stride=2
)   (32,8,8)
```


### Question 1

What is the output size after the first convolution?

Answer:

$$
32\times32\times16
$$

---

### Question 2

What is the output size after the first pooling layer?

Answer:

$$
16\times16\times16
$$

---

### Question 3

What is the output size after the second convolution?

Answer:

$$
16\times16\times32
$$

---

### Question 4

What is the output size after the second pooling layer?

Answer:

$$
8\times8\times32
$$

---

### Question 5

How many feature maps are produced by the first convolution?

Answer:

$$
16
$$

because:

$$
16\text{ filters}
\rightarrow
16\text{ feature maps}
$$

---

### Question 6

What happens if we change:

```python
stride=1
```

to:

```python
stride=2
```

Students should recognize:

> The feature map becomes smaller because the filter moves with larger steps.

---

# 11. Final Mental Model

End the session by returning to one diagram.

```text
                    IMAGE
                      │
                      ▼
                Small Filter
                      │
                      ▼
              Sliding Convolution
                      │
                      ▼
                 Feature Map
                      │
                      ▼
                    ReLU
                      │
                      ▼
                  Pooling
                      │
                      ▼
             Smaller Feature Map
                      │
                      ▼
              More Convolution
                      │
                      ▼
           Larger Receptive Field
                      │
                      ▼
          More Abstract Features
                      │
                      ▼
                Classification
```

---

# Key Takeaways

Students should leave the session remembering these statements:

### 1. Kernel

> A kernel is a small matrix of weights used in convolution.

### 2. Filter

> A filter is a learnable set of weights that operates across the input channels.

### 3. Feature Map

> Applying one filter across an image produces one feature map.

$$
\boxed{
1\text{ Filter}
\rightarrow
1\text{ Feature Map}
}
$$

### 4. Stride

> Stride determines how far the filter moves at each step.

### 5. Padding

> Padding allows us to control the spatial size of the feature map and preserve information near image boundaries.

### 6. Receptive Field

> The receptive field is the region of the original image that can influence a particular neuron.

### 7. Learning

> CNN filters are not manually designed. Their weights are learned through backpropagation and optimization.

$$
\boxed{
\text{Data}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradients}
\rightarrow
\text{Updated Filters}
}
$$

### 8. Depth

> Deeper layers combine local patterns into increasingly complex representations.

$$
\boxed{
\text{Edges}
\rightarrow
\text{Textures}
\rightarrow
\text{Shapes}
\rightarrow
\text{Object Parts}
\rightarrow
\text{Complex Patterns}
}
$$

